# Python 101 - Solutions
## Chapter VII

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_07.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
import random

import requests
from bs4 import BeautifulSoup

BASE_URI = './data/'

# A browser-ish user agent. Several of the sites below refuse python's default.
USER_AGENT = {'User-agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                             'AppleWebKit/537.36 (KHTML, like Gecko) '
                             'Chrome/140.0.0.0 Safari/537.36')}

### Loading the test page

The notebook serves `data/test.html` over `python -m http.server` so that `requests.get` has something real to fetch. For the solutions we just open the file, so this notebook runs with no server - the soup is identical either way.

```python
# what the notebook does:
response = requests.get('http://localhost:8000/data/test.html')
soup = BeautifulSoup(response.content, 'html.parser')
```

In [ ]:
with open(BASE_URI + 'test.html', encoding='utf-8') as handle:
    soup = BeautifulSoup(handle.read(), 'html.parser')

print(soup.title.get_text())

### Exercise: every **visible** headline and subtitle

The catch is `<section id="not_main_section" style="display: none;">` at the bottom of the page: it holds a fifth headline (`Fake true elements`) that a browser never shows. Searching the whole document finds 5 headlines; searching inside `#main_content` finds the 4 real ones.

This is the same trap as the `important` paragraphs earlier in the chapter, which is the point - **scope your search first, then select**.

In [ ]:
headline_tags = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']

main = soup.find(id='main_content')
visible = [tag.get_text() for tag in main.find_all(headline_tags)]

everything = [tag.get_text() for tag in soup.find_all(headline_tags)]

print('visible :', visible)
print('all     :', everything)

assert visible == ['Title of the page', 'Subtitle of the page',
                   'Sub-subtitle of the page', 'Sub-sub-subtitle of the page']
assert 'Fake true elements' in everything      # the hidden one
assert 'Fake true elements' not in visible

### 1. Save every important link to a file

Again scoped to `#main_content`, otherwise the hidden section's fake link sneaks in and you get 9 instead of 8.

In [ ]:
filename = 'important_urls.txt'

important_urls = []
for link in main.find_all('a'):
    href = link.get('href')
    if href and 'important_part' in href:
        important_urls.append(href)

with open(BASE_URI + filename, 'w', encoding='utf-8') as handle:
    for url in important_urls:
        handle.write(url + '\n')

print(len(important_urls), 'links saved')

written = open(BASE_URI + filename, encoding='utf-8').read().splitlines()
assert len(written) == 8, written
assert all('important_part' in url for url in written)
# the whole document would have given us the fake one too
assert len([a for a in soup.find_all('a')
            if 'important_part' in (a.get('href') or '')]) == 9

### 2. A random post from bash.hu

`http://bash.hu/random` actually returns a **page of 50** random posts, not one - so 'get a random post' means picking one of them.

In [ ]:
URI = "http://bash.hu/random"

response = requests.get(URI, headers=USER_AGENT)
response.raise_for_status()

soup_bash = BeautifulSoup(response.content, 'html.parser')
posts = soup_bash.find_all('div', class_='qtxt')

print(len(posts), 'posts on the page\n')
print(random.choice(posts).get_text().strip()[:300])

assert len(posts) > 0

### 3. The same thing, as a function

In [ ]:
def i_want_fun(output, times=5):
    collected = []
    for _ in range(times):
        response = requests.get(URI, headers=USER_AGENT)
        response.raise_for_status()
        posts = BeautifulSoup(response.content, 'html.parser').find_all('div', class_='qtxt')
        if posts:
            collected.append(random.choice(posts).get_text().strip())

    with open(output, 'w', encoding='utf-8') as handle:
        handle.write(('\n' + '-' * 60 + '\n').join(collected))

    return collected


collected = i_want_fun(BASE_URI + 'fun.txt', times=2)
print(len(collected), 'posts written to', BASE_URI + 'fun.txt')

assert len(collected) == 2
assert len(open(BASE_URI + 'fun.txt', encoding='utf-8').read()) > 0

### 4. And as a class

Note the notebook's own skeleton has `show_urls` in the class but calls `show_posts` further down. Both are defined below so either call works - worth pointing out to students as a real-life naming slip.

In [ ]:
class IWantFun:
    """Collects random posts from bash.hu."""

    URI = "http://bash.hu/random"

    def __init__(self):
        self.posts = []

    def crawl(self):
        """Fetch one random post and remember it."""
        response = requests.get(self.URI, headers=USER_AGENT)
        response.raise_for_status()
        found = BeautifulSoup(response.content, 'html.parser').find_all('div', class_='qtxt')
        if found:
            self.posts.append(random.choice(found).get_text().strip())
        return self

    def crawl_multiple(self, times=5):
        """Fetch `times` random posts."""
        for _ in range(times):
            self.crawl()
        return self

    def show_posts(self):
        """Print every post collected so far."""
        for index, post in enumerate(self.posts, start=1):
            print(f'--- {index} ---')
            print(post[:200])
        return self

    # the notebook's skeleton calls it show_urls in one place
    show_urls = show_posts

    def export(self, output):
        """Write the posts into a file."""
        with open(output, 'w', encoding='utf-8') as handle:
            handle.write(('\n' + '-' * 60 + '\n').join(self.posts))
        return self

    def reset(self):
        """Throw away everything collected so far."""
        self.posts = []
        return self


nine = IWantFun()
nine.crawl()
assert len(nine.posts) == 1

nine.crawl_multiple(2)
assert len(nine.posts) == 3

nine.export(BASE_URI + 'fun.txt')
assert len(open(BASE_URI + 'fun.txt', encoding='utf-8').read()) > 0

nine.reset()
assert nine.posts == []
print('crawl / crawl_multiple / export / reset all behave')

In [ ]:
# tidy up the files this notebook created
import os

for name in ('important_urls.txt', 'fun.txt'):
    path = BASE_URI + name
    if os.path.exists(path):
        os.remove(path)
print('cleaned up')